# Library exploration

Basic GLiNER inference and PyMuPDF extraction, following the libraries' quick starts.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / 'ner').is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from ner.evaluation import ENTITY_LABELS, evaluate_ner_strict, evaluate_ner_relaxed
from ner.utils import load_jsonl, save_responses_to_jsonl
from ner.inference import load_gliner_model
model = load_gliner_model()


In [2]:
text = 'Ada Example works at Example Institute in London.'
entities = model.predict_entities(text, ['person', 'organization', 'location'], threshold=0.5)
entities

[{'start': 0,
  'end': 11,
  'text': 'Ada Example',
  'label': 'person',
  'score': 0.9774747490882874},
 {'start': 21,
  'end': 38,
  'text': 'Example Institute',
  'label': 'organization',
  'score': 0.9742453694343567},
 {'start': 42,
  'end': 48,
  'text': 'London',
  'label': 'location',
  'score': 0.9888968467712402}]

## What is a span?

`start` is the first character index and `end` is exclusive: `text[start:end]` gives the predicted mention. Scores express model confidence; labels are the entity types requested in the call.

In [3]:
[(entity['start'], entity['end'], text[entity['start']:entity['end']]) for entity in entities]

[(0, 11, 'Ada Example'), (21, 38, 'Example Institute'), (42, 48, 'London')]

In [4]:
other_text = 'The meeting at Example University is on 15 November 2026.'
model.predict_entities(other_text, ['organization', 'date'], threshold=0.5)

[{'start': 15,
  'end': 33,
  'text': 'Example University',
  'label': 'organization',
  'score': 0.9950730204582214},
 {'start': 40,
  'end': 56,
  'text': '15 November 2026',
  'label': 'date',
  'score': 0.9975318908691406}]

In [5]:
import pymupdf
path = ROOT / 'data/pdf_clean/test-0001.pdf'
with pymupdf.open(path) as document:
    print('Pages:', len(document))
    print('Characters extracted per page:', [len(page.get_text()) for page in document])

Pages: 1
Characters extracted per page: [4090]


The PDF text is kept local. An image-only PDF requires OCR before NER can detect entities. This project measures the requested direct extraction pipeline without adding OCR.